### 2) K-Nearest Neighbors (KNN) Nedir?

KNN, şu ana kadar gördüğümüz modellerden tamamen farklı bir mantıkla çalışır hiçbir formül/katsayı öğrenmez, bunun yerine "bana arkadaşını söyle, sana kim olduğunu söyleyeyim" mantığıyla çalışır.

**Temel Mantık:**
Yeni bir veri noktası geldiğinde, KNN:
1. Bu noktaya en yakın K tane komşuyu (eğitim verisindeki) bulur
2. Bu K komşunun çoğunluk oyuna bakar (sınıflandırmada)
3. Çoğunluğun sınıfını, yeni noktaya atar

**"Yakınlık" Nasıl Ölçülür? — Öklid Mesafesi**
- Mesafe = √[(x1-x2)² + (y1-y2)² + ...]

**K Değerinin Önemi (Bias-Variance Tradeoff)**
- K küçük (örn. K=1): Model çok "hassas" olur, tek bir komşuya göre karar verir düşük bias, yüksek variance (overfitting/ezberleme riski)
- K büyük (örn. K=200): Model çok "genellemeci" olur yüksek bias, düşük variance (underfitting riski)
- En iyi K değeri, GridSearchCV gibi tekniklerle bulunur (Faz 6'da detaylandırılacak)

**Neden Feature Scaling KNN'de Kritik?**
KNN mesafeye dayalı çalıştığı için, ölçek farkları büyük sorun yaratır. Örneğin yaş (0-100) ile maaş (0-100.000) birlikte kullanılırsa, maaştaki farklar mesafeyi domine eder. Scaling (z-skor), tüm sütunları aynı ölçeğe getirerek bu adaletsizliği giderir.

### KNN'in Varsayımları
1. Benzer noktalar, benzer sonuç verir (temel varsayım)
2. Tüm özellikler aynı ölçekte olmalı (scaling şart)
3. Düşük boyutluluk tercih edilir (Curse of Dimensionality)
4. Gürültüsüz/temiz veri varsayımı

### Avantajlar
- Basit ve sezgisel, eğitim süreci yok denecek kadar az
- Varsayımsız (non-parametric), doğrusal olmayan örüntüleri yakalayabilir
- Hem sınıflandırma hem regresyon için kullanılabilir
- Yeni veriye kolay adapte olur

### Dezavantajlar
- Yavaş tahmin süresi (her tahminde tüm veriyle mesafe hesaplanır)
- Feature Scaling zorunluluğu
- Curse of Dimensionality
- Dengesiz veri setlerinde zayıf
- K seçimi kritik, tuning gerektirir
- Hafıza yoğun

In [2]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Veri setini yükle (temizlenmiş titanic DataFrame'i hazır olduğunu varsayıyoruz)
titanic = sns.load_dataset("titanic")
# 2. age sütununu grup bazlı medyan ile doldur (pclass + sex kombinasyonuna göre)
titanic['age'] = titanic.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# 3. deck sütununu çıkarıyoruz. (çok fazla eksik veri, %77)
titanic = titanic.drop(columns=['deck'])

# 4. embark_town ve embarked'daki birkaç eksik satırı silelim.
titanic = titanic.dropna(subset=['embark_town'])

# 5. alive (survived'ın kopyası) ve embarked (embark_town'ın kısaltması) sütunlarını çıkarıyoruz.
titanic = titanic.drop(columns=['alive', 'embarked'])

# 6. Kategorik sütunları One-Hot Encoding ile çevirelim. drop_first (multicollinearity olmasını engelliyoruz.)
titanic = pd.get_dummies(titanic, columns=['embark_town', 'sex'], drop_first=True)

# 7. adult_male'i int'e çevirelim.
titanic['adult_male'] = titanic['adult_male'].astype(int)

# 8. class sütununu ordinal (sıralı) olarak encode ediyoruz.
sinif_siralamasi = {'First': 1, 'Second': 2, 'Third': 3}
titanic['class'] = titanic['class'].map(sinif_siralamasi)

# 9. who ve class sütunlarını çıkarıyoruz. (redundant bilgi taşıyorlardı)
titanic = titanic.drop(columns=['who', 'class'])

# 10. Kalan tüm bool sütunları int'e çevir
bool_sutunlar = titanic.select_dtypes(include='bool').columns
titanic[bool_sutunlar] = titanic[bool_sutunlar].astype(int)

X = titanic.drop(columns=['survived'])
y = titanic['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling — KNN için zorunlu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# K=5 ile model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Farklı K değerlerini karşılaştır
print("\n--- K Karşılaştırması ---")
for k in [1, 3, 5, 7, 9, 15, 25, 50]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_k)
    print(f"K={k}: Accuracy={acc:.4f}")

Accuracy: 0.7753
Confusion Matrix:
[[86 23]
 [17 52]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       109
           1       0.69      0.75      0.72        69

    accuracy                           0.78       178
   macro avg       0.76      0.77      0.77       178
weighted avg       0.78      0.78      0.78       178


--- K Karşılaştırması ---
K=1: Accuracy=0.7303
K=3: Accuracy=0.7865
K=5: Accuracy=0.7753
K=7: Accuracy=0.7753
K=9: Accuracy=0.7921
K=15: Accuracy=0.8146
K=25: Accuracy=0.8034
K=50: Accuracy=0.8090


### Sonuçların Yorumlanması

K=5 ile model %77.53 accuracy verdi. Farklı K değerleriyle yapılan karşılaştırmada:

- K=1: Accuracy=0.7191 (en düşük) — model tek komşuya göre karar verdiği için aşırı hassas/kararsız davrandı, bu düşük bias ama yüksek variance (ezberleme/overfitting) durumudur.
- K arttıkça (1→15): Accuracy genel olarak yükseldi, K=15'te en yüksek değere (0.8146) ulaşıldı. Bu, K=1'deki aşırı ezberlemenin (yüksek variance) zararının, K artınca azalmasının net performansı iyileştirdiğini gösteriyor — model daha "sakin" ve genellenebilir hale geldi.
- K=25, K=50: Accuracy hafifçe geriledi (0.8034, 0.8090) — çok fazla komşuya bakmak, modelin aşırı basitleşip (yüksek bias) bazı ayrımları kaçırmasına yol açmaya başlıyor olabilir.

**Logistic Regression ile Karşılaştırma:** Logistic Regression %82.58 accuracy vermişti. KNN'in en iyi sonucu (K=15, %81.46) buna çok yakın — aralarında sadece ~%1'lik fark var. Bu, iki farklı algoritmanın (biri formül tabanlı, diğeri mesafe tabanlı) bu veri setinde benzer başarı seviyesine ulaşabildiğini gösteriyor.

**Genel Ders:** K değeri, Bias-Variance Tradeoff'un KNN'deki somut karşılığıdır — "az kişiye danışmak" (küçük K) kararsızlık, "çok kişiye danışmak" (büyük K) sıradanlaşma riski taşır; en iyi K, ikisi arasındaki dengeyi bulan noktadır.